# 🛸 Video Colorizer (Lost in Space) - Google Colab (GPU NVIDIA)

Official notebook for running the industrial *Lost in Space* colorization pipeline in the cloud with CUDA GPU acceleration (NVIDIA T4, L4, V100, or A100).

Repository: [github.com/jdmarinv/video-colorizer](https://github.com/jdmarinv/video-colorizer)

### Main features:
- **Full luminance preservation ($L$ in CIE LAB):** no loss of film grain or microcontrast.
- **Optical-flow temporal propagation:** suppresses color boiling.
- **Canonical palette:** rules derived from color Seasons 2 and 3.
- **Google Drive integration:** direct video input and output without consuming local storage.

## Step 1: Verify GPU Acceleration (NVIDIA CUDA)
Select a GPU runtime from the menu:  
`Runtime` > `Change runtime type` > **T4 GPU** (or better).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU is active. Open 'Runtime' > 'Change runtime type' and select a GPU.")

## Step 2: Mount Google Drive
Connect Google Drive to read input `.mkv` files and save completed colorized episodes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Suggested Google Drive directories
drive_base = Path('/content/drive/MyDrive/LostInSpace')
input_dir = drive_base / 'Input'
output_dir = drive_base / 'Colorized'

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Drive input directory: {input_dir}")
print(f"✅ Drive output directory:  {output_dir}")
print("💡 Place black-and-white episodes here (.mkv or .mp4).")

## Step 3: Configure the Repository and Dependencies
Clone the project from `github.com/jdmarinv/video-colorizer`, or use an existing copy from Drive.

In [ ]:
#@title ⚙️ Initialize the Colorizer { display-mode: "form" }
source_mode = "Clone from GitHub / Copy repository" #@param ["Clone from GitHub / Copy repository", "Use existing Google Drive directory"]
github_repo_url = "https://github.com/jdmarinv/video-colorizer.git" #@param {type:"string"}
github_token = "" #@param {type:"string"} {help: "Optional: enter a Personal Access Token (PAT) when the repository is private"}
drive_project_path = "/content/drive/MyDrive/video-colorizer" #@param {type:"string"}

import os
import subprocess
from pathlib import Path

work_dir = Path("/content/video-colorizer")

if source_mode == "Use existing Google Drive directory":
    if Path(drive_project_path).exists():
        work_dir = Path(drive_project_path)
        print(f"Using project directly from Drive: {work_dir}")
    else:
        print(f"⚠️ Could not find {drive_project_path}; cloning instead...")
        clone_url = github_repo_url
        if github_token.strip():
            clone_url = github_repo_url.replace("https://", f"https://{github_token.strip()}@")
        !git clone {clone_url} /content/video-colorizer
        work_dir = Path("/content/video-colorizer")
else:
    if not work_dir.exists():
        clone_url = github_repo_url
        if github_token.strip():
            clone_url = github_repo_url.replace("https://", f"https://{github_token.strip()}@")
        !git clone {clone_url} /content/video-colorizer
    else:
        print("The repository already exists at /content/video-colorizer. Updating with git pull...")
        !git -C /content/video-colorizer pull

%cd {work_dir}

# Install Python dependencies
!pip install -q opencv-python tqdm

# Create local working directories
!mkdir -p models live_previews

# Download large-model weights (912 MB) when absent
model_large = Path("models/ddcolor_modelscope.pth")
if not model_large.exists():
    print("Downloading the DDColor neural model (912 MB)... This takes about 30 seconds in Colab.")
    !curl -L https://huggingface.co/piddnad/DDColor-models/resolve/main/ddcolor_modelscope.pth -o models/ddcolor_modelscope.pth
else:
    print("✅ DDColor model already downloaded.")

print("\n🎉 Environment configured and ready.")

## Step 4: Run Colorization from the Form
Set the desired parameters and click Play to begin GPU-accelerated processing.

In [ ]:
#@title 🚀 Run Episode Colorizer { display-mode: "form" }
target = "1" #@param {type:"string"} {help: "Episode number (for example, '1'), range ('1-3'), or 'all'"}
mode = "balanced" #@param ["balanced", "fast", "direct"]
sample_step = 8 #@param {type:"integer"}
crf = 18 #@param {type:"slider", min:14, max:28, step:1}
preset = "medium" #@param ["ultrafast", "fast", "medium", "slow"]
model_size = "large" #@param ["large", "tiny"]
chunk_size = 500 #@param {type:"integer"}
force = False #@param {type:"boolean"}
input_dir = "/content/drive/MyDrive/LostInSpace/Input" #@param {type:"string"}
output_dir = "/content/drive/MyDrive/LostInSpace/Colorized" #@param {type:"string"}

cmd = [
    "python", "colorize_episode.py", target,
    "--mode", mode,
    "--input-dir", input_dir,
    "--output-dir", output_dir,
    "--crf", str(crf),
    "--preset", preset,
    "--model-size", model_size,
    "--chunk-size", str(chunk_size)
]

if sample_step:
    cmd.extend(["--sample-step", str(sample_step)])
if force:
    cmd.append("--force")

print("Command:", " ".join(cmd))
!{" ".join(cmd)}

## Step 5: Preview Results and the Live Gallery
Inspect preview frames generated in real time inside `live_previews/`.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import glob

preview_files = sorted(glob.glob("live_previews/*.jpg"))
if preview_files:
    print(f"Showing latest live preview: {preview_files[-1]}")
    display(Image(preview_files[-1]))
else:
    print("No previews are available in live_previews/ yet.")